# Runtime  运行时
LangChain 的 create_agent 底层运行在 LangGraph 的运行时环境中。

Runtime:
1. **Context:** 静态信息，例如用户 ID、数据库连接或其他代理调用依赖项。
2. **Store:** 用于长期记忆的 BaseStore 实例
3. **Stream writer:** 用于通过 "custom" 流模式传输信息的对象

## Access
使用 create_agent 创建代理时，您可以指定` context_schema `来定义存储在代理 Runtime 中的 context 结构。

调用代理时，请传递包含运行相关配置的 context 参数：

In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

base_url = os.getenv("QWEN_BASE_URL")
api_key = os.getenv("QWEN_API_KEY")

model = ChatOpenAI(
    model="qwen3.5-plus-2026-02-15",
    base_url=base_url,
    api_key=api_key,
)

In [3]:
from dataclasses import dataclass

from langchain.agents import create_agent


@dataclass
class Context:
    user_name: str

agent = create_agent(
    model,
    # tools=[...],
    context_schema=Context  
)

agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    context=Context(user_name="John Smith")
)

{'messages': [HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}, id='4b33778d-55ad-42f4-8405-bd3d6e4f7904'),
  AIMessage(content="I don't actually know your name! I'm an AI assistant, so I don't have access to your personal information.\n\nBut if you'd like to tell me, I'd be happy to call you by it during our conversation.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 762, 'prompt_tokens': 15, 'total_tokens': 777, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 709, 'rejected_prediction_tokens': None, 'text_tokens': 762}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 15}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus-2026-02-15', 'system_fingerprint': None, 'id': 'chatcmpl-3bab77bb-daf7-9572-b236-9925fd4b9729', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d4203-5aeb-7703-a64e-56856

### Inside tools  内部工具
可以通过访问工具内部的运行时信息来执行以下操作：
- 获取上下文
- 读取或写入长期记忆
- 写入自定义流 （例如，工具进度/更新）
使用 ToolRuntime 参数可以访问工具内部的 Runtime 对象。

In [ ]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime  

@dataclass
class Context:
    user_id: str

@tool
def fetch_user_email_preferences(runtime: ToolRuntime[Context]) -> str:
    """Fetch the user's email preferences from the store."""
    user_id = runtime.context.user_id  

    preferences: str = "The user prefers you to write a brief and polite email."
    if runtime.store:
        if memory := runtime.store.get(("users",), user_id):
            preferences = memory.value["preferences"]

    return preferences

### Inside middleware  中间件内部
可以访问中间件中的运行时信息，以根据用户上下文创建动态提示、修改消息或控制代理行为。

在节点式钩子中，可以使用 Runtime 参数访问 Runtime 对象。对于包装式钩子 ， Runtime 对象可通过 ModelRequest 参数访问。

In [4]:
from dataclasses import dataclass

from langchain.messages import AnyMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import dynamic_prompt, ModelRequest, before_model, after_model
from langgraph.runtime import Runtime


@dataclass
class Context:
    user_name: str

# Dynamic prompts
@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context.user_name  
    system_prompt = f"You are a helpful assistant. Address the user as {user_name}."
    return system_prompt

# Before model hook
@before_model
def log_before_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    print(f"Processing request for user: {runtime.context.user_name}")
    return None

# After model hook
@after_model
def log_after_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    print(f"Completed request for user: {runtime.context.user_name}")
    return None

agent = create_agent(
    model,
    # tools=[...],
    middleware=[dynamic_system_prompt, log_before_model, log_after_model],
    context_schema=Context
)

agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    context=Context(user_name="John Smith")
)

Processing request for user: John Smith
Completed request for user: John Smith


{'messages': [HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}, id='f36b3189-27a7-4d50-8815-4495af07e678'),
  AIMessage(content='Your name is John Smith.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 479, 'prompt_tokens': 33, 'total_tokens': 512, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 468, 'rejected_prediction_tokens': None, 'text_tokens': 479}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 33}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus-2026-02-15', 'system_fingerprint': None, 'id': 'chatcmpl-c3f92fcd-4cad-937e-be68-79a114bfbbcc', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d420a-a66b-7000-9bf0-25cbc6126e12-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 479, 'total_tokens': 512, 'input_token_details': {}, 'output_token_d